### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import sys
sys.path.append('./utils')
from svg_processor import SVGSanitizer, SVGProcessor,svg_constraints
from siglip_class import SVGMetricEvaluator

### Random seed for reproducibility

In [3]:
import torch
import random
import numpy as np
import multiprocessing as mp

mp.set_start_method("spawn", force=True)
# Now set your seed
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [4]:
import concurrent
import io
import logging
import re
import re2
import cairosvg
import kagglehub
from lxml import etree
from vllm import LLM, SamplingParams
import gc


class Model:
    
    def __init__(self):

        self.model_path="./lora/Qwen25_7B_Instruct_lora_fp16_r256_s2000_i1000_msl2048_awq"
        self.model = LLM(
            model=self.model_path,
            max_model_len=1024,
            gpu_memory_utilization=0.85,
            dtype="half",
            seed=123,
            disable_log_stats=True
        )

       
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model 
        gc.collect()     


    def _format_prompt(self, description: str) -> str:
        return  f"""Below is an instruction that describes a task, paired with an input that provides further context. 
                Write a response that appropriately completes the request.
                
                ### Instruction:
                Generate a SVG code for the given input:
                
                ### Input:
                {description}
                
                ### Response:
                """
    
    def get_response(self, descriptions):
        
        formatted_input = [self._format_prompt(desc) for desc in descriptions]
        sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=1024,n=1)
        outputs = self.model.generate(formatted_input, sampling_params)
        
        #suitable for batch inputs as well
        output_list=[]
        for output in outputs:
            prompt = output.prompt
            generated_text = output.outputs[0].text
            output_list.append(generated_text.strip())
        return output_list
    
    def predict(self, descriptions: list[str], max_new_tokens=1024) -> list[str]:
        output_decoded_list = self.get_response(descriptions)
        final_svg_code_list = []
    
        for description, output in zip(descriptions, output_decoded_list):
            base_svg = SVGProcessor.clean_and_extract_svgs(output, self.default_svg)
            clean_svg = self.sanitizer.enforce_constraints(base_svg)
            final_svg = SVGProcessor.svg_conversion_check(description, clean_svg, self.default_svg)
            final_svg_code_list.append(final_svg)
    
        return final_svg_code_list


INFO 04-22 22:38:28 [__init__.py:239] Automatically detected platform cuda.


In [5]:
model=Model()

WARNING 04-22 22:38:29 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 04-22 22:38:34 [config.py:585] This model supports multiple tasks: {'classify', 'reward', 'embed', 'generate', 'score'}. Defaulting to 'generate'.
INFO 04-22 22:38:35 [awq_marlin.py:114] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 04-22 22:38:35 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-22 22:38:35 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='./lora/Qwen25_7B_Instruct_lora_fp16_r256_s2000_i1000_msl2048_awq', speculative_config=None, tokenizer='./lora/Qwen25_7B_Instruct_lora_fp16_r256_s2000_i1000_msl2048_awq', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_c

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 04-22 22:38:37 [loader.py:447] Loading weights took 1.42 seconds
INFO 04-22 22:38:38 [gpu_model_runner.py:1186] Model loading took 5.2048 GB and 1.880183 seconds
INFO 04-22 22:38:45 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/ffcaed2437/rank_0_0 for vLLM's torch.compile
INFO 04-22 22:38:45 [backends.py:425] Dynamo bytecode transform time: 7.25 s
INFO 04-22 22:38:46 [backends.py:115] Directly load the compiled graph for shape None from the cache
INFO 04-22 22:38:50 [monitor.py:33] torch.compile takes 7.25 s in total
INFO 04-22 22:38:52 [kv_cache_utils.py:566] GPU KV cache size: 33,376 tokens
INFO 04-22 22:38:52 [kv_cache_utils.py:569] Maximum concurrency for 1,024 tokens per request: 32.59x
INFO 04-22 22:39:09 [gpu_model_runner.py:1534] Graph capturing finished in 17 secs, took 0.57 GiB
INFO 04-22 22:39:09 [core.py:151] init engine (profile, create kv cache, warmup model) took 31.13 seconds


In [6]:
#tmp=model.predict(['sun rising in the east','A golden goose with a fish'])

In [7]:
#print(tmp[0])

In [8]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/svg_score_test_vqa.csv',header=[0])
print(df.shape)
df.head(2)

(75, 7)


,description,gpt_svg,gpt_score_sl,response,vqa_pair,response_2,gpt_svg_2
0,"'Vibrant autumn forest',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.904117,Here is the visual question answering (VQA) pa...,"{'description': 'Vibrant autumn forest', 'ques...","Here's an improved SVG representation of a ""Vi...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
1,"'Morning dew on grass',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.987024,Here is a visual question answering (VQA) pair...,"{'description': 'Morning dew on grass', 'quest...","Here's an improved SVG representation of ""Morn...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


In [9]:
# from tqdm import tqdm
# tqdm.pandas()
# df['svg_3'] = df['description'].progress_apply(lambda x: model.predict(x))

In [10]:
from tqdm import tqdm
description_list = [s.strip(" ',") for s in df['description'].to_list()]
batch_size = 12
results = []

for i in tqdm(range(0, len(description_list), batch_size), desc="Batch prediction"):
    batch = description_list[i:i + batch_size]
    batch_result = model.predict(batch)  # Ensure this handles a list of inputs
    results.extend(batch_result)


Batch prediction:   0%|                                   | 0/7 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/12 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   8%| | 1/12 [00:05<00:55,  5.05s/it, est. speed input: 11.89
cessed prompts:  17%|▏| 2/12 [00:05<00:24,  2.50s/it, est. speed input: 20.66
cessed prompts:  25%|▎| 3/12 [00:06<00:16,  1.85s/it, est. speed input: 26.05
cessed prompts:  33%|▎| 4/12 [00:07<00:11,  1.43s/it, est. speed input: 31.12
cessed prompts:  42%|▍| 5/12 [00:08<00:09,  1.31s/it, est. speed input: 34.04
cessed prompts:  58%|▌| 7/12 [00:09<00:03,  1.34it/s, est. speed input: 45.77
Processed prompts: 100%|█| 12/12 [00:10<00:00,  1.17it/s, est. speed input: 70.3
ERROR:root:SVG Parse Error: Unescaped '<' not allowed in attributes values, line 48, column 61 (<string>, line 48). Returning default SVG.
ERROR:root:SVG Parse Error: Unescaped '<' not allowed in attributes values, line 1, column 2505 (<string>, line 1). Returning default SVG.
ERROR:root:SVG

In [13]:
df['svg_3']=results

In [12]:
model.close_model()

In [14]:
#SigLip Score
tqdm.pandas()
evaluator = SVGMetricEvaluator()
df['svg_score_3'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg_3']), axis=1)

model.safetensors:   0%|          | 0.00/3.51G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.40M [00:00<?, ?B/s]

100%|███████████████████████████████████████████| 75/75 [00:04<00:00, 15.42it/s]


In [15]:
df['svg_score_3'].mean()

0.6059735314523585